# CL2: full-corpus BERT on Colab

Fine-tunes pretrained BERT on **all 3,085,832 lexical examples**, with the same ten conversation folds and separate binary/ten-class models. Two complete training epochs per fold. Inputs are transcript contexts only.

The Mac checkpoint is preserved locally. Colab initializes the first missing fold from pretrained BERT because the large checkpoint transfer timed out. Future Colab interruptions resume automatically from persistent Drive checkpoints.

The private **CL2 Full Corpus BERT** folder contains `project.tar.gz` (97 MB) and `colab_resume.py` (small training launcher). Results and rolling checkpoint backups remain in that folder. Run cells in order with an available GPU runtime; no purchase or plan upgrade is required.

In [ ]:
import torch
assert torch.cuda.is_available(), "Choose a GPU runtime first"
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import tarfile, hashlib, json, sys
BASE = Path('/content/drive/MyDrive/CL2 Full Corpus BERT')
WORK = Path('/content/cl2_full_bert')
archive = BASE/'project.tar.gz'
assert archive.exists(), 'The project archive is not in the dedicated Drive folder yet'
checksum = hashlib.sha256()
with archive.open('rb') as stream:
    for block in iter(lambda: stream.read(8*1024*1024), b''):
        checksum.update(block)
assert checksum.hexdigest() == 'bbc672cd581f105359192f176a7b93dc74fb689e39a2235ee60abb9f0b58853a', 'Archive checksum mismatch'
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall(WORK, filter='data')
manifest = json.loads((WORK/'data/classification/full/manifest.json').read_text())
assert manifest['examples'] == 3085832
import shutil
helper = BASE/'colab_resume.py'
assert helper.exists(), 'The training launcher has not finished uploading'
assert hashlib.sha256(helper.read_bytes()).hexdigest() == '2357d8ed7ad8e5901d1f09d44fc74dba0fc62b627a0b55330e2081e040bbe068', 'Launcher checksum mismatch'
shutil.copy2(helper, WORK/'scripts/colab_resume.py')
print('Prepared full corpus:', manifest['examples'], 'examples')

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'transformers==5.22.2', 'scikit-learn==1.9.1', 'scipy', 'pandas', 'pyyaml', 'jiwer'])

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, str(WORK/'scripts/colab_resume.py'),
                       '--drive-root', str(BASE), '--work', str(WORK), '--from-pretrained'])